``get_labels_quantile`` discretizes a continuous target into a binary label array at a single quantile cut (default median): samples at or above the cut are the test group. This bridges regression-style targets (thermostability, binding affinity) into the binary CPP framework:

In [1]:
import numpy as np
import aaanalysis as aa

targets = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
aa.SequenceFeature.get_labels_quantile(targets, q=0.5)

array([0, 0, 0, 1, 1, 1])

Use the labels directly with ``CPP.run``:

In [2]:
df_seq = aa.load_dataset(name="DOM_GSEC", n=8)
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)

labels = aa.SequenceFeature.get_labels_quantile(np.linspace(0, 1, len(df_parts)), q=0.5)
df_feat = aa.CPP(df_parts=df_parts).run(labels=labels, n_filter=4)
aa.display_df(df_feat[["feature", "abs_auc"]], n_rows=10, show_shape=True)

1. CPP creates 580140 features for 16 samples
1.1 Assigning scale values to parts
   |                         | 0.0%

   |........                 | 33.3%

   |................         | 66.7%

   |.........................| 100.0%


1.2 Streaming pre-filter stats (mask in stream)
   |                         | 0.0%

...

   |.........................| 100.0%
2. CPP pre-filters 29007 features (5.0%) with highest 'abs_mean_dif' and 'max_std_test' <= 0.2 (kept=517501 of 580140)


3. CPP filtering algorithm


4. CPP returns df of 4 unique features with general information and statistics


DataFrame shape: (4, 2)


,feature,abs_auc
1,"TMD-Pattern(C,4,8)-NAKH900112",0.500000
2,"TMD_C_JMD_C-Pat...4,8)-NAKH900112",0.500000
3,"TMD-Pattern(C,4,8)-AURR980114",0.500000
4,"TMD_C_JMD_C-Pat...4,8)-AURR980114",0.500000


**What can go wrong?** A constant target (or a cut that leaves one side empty) yields a single class and is rejected up front:

In [3]:
try:
    aa.SequenceFeature.get_labels_quantile([5.0, 5.0, 5.0], q=0.5)
except ValueError as e:
    print("ValueError:", e)

ValueError: 'targets' produce a single class at q=0.5 (all values equal, or the cut leaves one side empty); adjust 'q' or 'targets'.


**Further parameters.** ``label_test`` / ``label_ref`` set the integer values assigned to the at-or-above-cut (test) and below-cut (reference) groups:

In [4]:
labels_q = aa.SequenceFeature.get_labels_quantile([1.0, 2.0, 3.0, 4.0], q=0.5,
                                                  label_test=1, label_ref=0)
print(labels_q)

[0 0 1 1]
